In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xarray as xr

import eradiate
from eradiate.data.convert import aer_v1_to_aer_core_v2, libradtran_to_aer_core_v2

sns.set_style("ticks")
eradiate.set_mode("mono")

In [ ]:
ds_desert_aer_v1 = eradiate.fresolver.load_dataset(
    "aerosol/govaerts_2021-desert-extrapolated.nc"
)
ds_wc_libradtran = ds = xr.open_dataset("wc.sol.mie.cdf").sel(nreff=0)

In [ ]:
display(ds_desert_aer_v1)
ds_desert_aer_core_v2 = aer_v1_to_aer_core_v2(ds_desert_aer_v1, phase_scale=4.0 * np.pi)
display(ds_desert_aer_core_v2)
integral = np.array(
    [
        ds_desert_aer_core_v2.isel(phamat=0, w=iw, drop=True)["phase"].integrate("mu")
        for iw in range(ds_desert_aer_core_v2.sizes["w"])
    ]
)
print(integral)

In [ ]:
display(ds_wc_libradtran)
ds_wc_aer_core_v2 = libradtran_to_aer_core_v2(ds_wc_libradtran)
display(ds_wc_aer_core_v2)
integral = np.array(
    [
        ds_wc_aer_core_v2.isel(phamat=0, w=iw)["phase"].integrate("mu")
        for iw in range(ds_wc_aer_core_v2.sizes["w"])
    ]
)
print(f"{integral = }")

In [ ]:
ds_wc_aer_core_v2["pmom"].isel(phamat=0).plot(x="w")

In [ ]:
iw = 0
data = {
    "desert-aer_v1": (
        ds_desert_aer_v1["mu"].values,
        ds_desert_aer_v1["phase"].isel(i=0, j=0, w=iw).values * 4.0 * np.pi,
        {},
    ),
    "desert-aer_core_v2": (
        ds_desert_aer_core_v2["mu"].isel(w=iw).values,
        ds_desert_aer_core_v2["phase"].isel(phamat=0, w=iw).values,
        {"ls": "", "marker": ".", "markersize": 2},
    ),
    "wc-libradtran": (
        np.cos(np.deg2rad(ds_wc_libradtran["theta"].isel(nphamat=0, nlam=iw).values)),
        ds_wc_libradtran["phase"].isel(nphamat=0, nlam=iw).values,
        {},
    ),
    "wc-aer_core_v2": (
        ds_wc_aer_core_v2["mu"].isel(w=iw).values,
        ds_wc_aer_core_v2["phase"].isel(phamat=0, w=iw).values,
        {"ls": "", "marker": ".", "markersize": 2},
    ),
}

fig, ax = plt.subplots()

for label, (x, y, kwargs) in data.items():
    ax.plot(x, y, label=label, **kwargs)

ax.set_yscale("log")
ax.legend()

In [ ]:
from eradiate.radprops._particles import ParticleProperties
from eradiate.units import unit_registry as ureg

props = ParticleProperties(ds_wc_aer_core_v2)
display(props.pmom)
print(f"{props.eval_ssa(550. * ureg.nm) = }")
print(f"{props.eval_ext(550. * ureg.nm) = }")
print(f"{props.eval_phase(550. * ureg.nm) = }")
print(f"{props.eval_pmom(550. * ureg.nm) = }")

In [ ]:
import drjit as dr
import mitsuba as mi

mi.set_variant("scalar_mono_double")

mus_a = np.linspace(-1, 1, 11)
values_a = np.linspace(0, 1, 11)
mus_b = np.linspace(-1, 1, 21)
values_b = np.ones_like(mus_b)

mus_init = np.linspace(-1, 1, np.max((len(mus_a), len(mus_b))))
values_init = np.ones_like(mus_init)

# Bug dans tabphase_polarized ? Est-ce que les valeurs des nœuds de la distribution
# sont bien mises à jour quand on fait une update ?
p = mi.load_dict(
    {
        "type": "tabphase_polarized",
        "m11": ",".join(map(str, values_init)),
        "nodes": ",".join(map(str, mus_init)),
    }
)
params = mi.traverse(p)

values_update = np.zeros_like(values_init)
values_update[: len(mus_a)] = values_a
mus_update = np.zeros_like(mus_init)
mus_update[: len(mus_a)] = mus_a
mus_update[len(mus_a) :] = np.linspace(1.00001, 2, len(mus_init) - len(mus_a))

params["nodes"] = mus_update
params["m11"] = values_update
params.update()
print(p)

ctx = dr.zeros(mi.PhaseFunctionContext)
mei = mi.MediumInteraction3f()
mei.wi = (0, 0, 1)
wo = mi.Vector3f(0, 0, 1)
p.eval_pdf(ctx, mei, wo)